In [2]:
from itertools import product
import math
import os
import random
from typing import Tuple, Optional

import cv2
import numpy as np

In [97]:
DATA_PATH  = os.path.join('.', 'data')
PHOTO_PATH = os.path.join(DATA_PATH, 'photo') 
FONT_PATH  = os.path.join(DATA_PATH, 'fonts')

P_TEST = .2

In [3]:
#img = cv2.imread('HR_result.png')

In [4]:
#img  = cv2.imread(os.path.join(DATA_PATH, '1.jpg'), cv2.IMREAD_UNCHANGED)
img  = cv2.imread(os.path.join(PHOTO_PATH, '7.jpg'), cv2.IMREAD_UNCHANGED)
# black font and many scratches
font = cv2.imread(os.path.join(FONT_PATH, 'font_15.jpg'), cv2.IMREAD_UNCHANGED)
# smash papper
#font = cv2.imread(os.path.join(FONT_PATH, 'font_2.jpg'), cv2.IMREAD_UNCHANGED) 

Варианты искажений:   
- [ ] кофе чашка;   
      - ресайз со смещением и поворотом;
      - вырезка со смещением и поворотом;
- [ ] кофе пролито;   
- [ ] царапины белые:   
      - ресайз со смещением и поворотом;   
      - вырезка со смещением и поворотом;   
- [ ] царапины черные:   
      - ресайз со смещением и поворотом;   
      - вырезка со смещением и поворотом;   
-   

In [79]:
tmp_all = os.listdir(FONT_PATH)

scratch = [el for el in tmp_all if el.startswith('scratch')]
fold = [el for el in tmp_all if el.startswith('fold')]

cup = [el for el in tmp_all if el.startswith('cup')]
coffee = [el for el in tmp_all if el.startswith('coffee')]

In [125]:
all_photo = os.listdir(PHOTO_PATH)

for photo in all_photo:
    #choice = random.choice(['scratch', 'fold', 'cup', 'coffee'])
    choice = random.choice(['scratch', 'cup',])
    random.choices(['train', 'test'], [1 - p_test, p_test])     # ?val?

    if choice == 'scratch':
        color = random.choice(['white', 'black'])
        distortion = random.choice(scratch)

    if choice == 'fold':
        distortion = random.choice(fold)

    if choice == 'cup':
        distortion = random.choice(cup)

    if choice == 'coffee':
        distortion = random.choice(coffee)

    font = cv2.imread(os.path.join(FONT_PATH, distortion), cv2.IMREAD_UNCHANGED)
    img = cv2.imread(os.path.join(FONT_PATH, photo), cv2.IMREAD_UNCHANGED)

    

In [47]:
def do_something(inp_img: np.ndarray, inp_shape: Tuple[int, int]) -> np.ndarray:
    '''
    '''
    h = inp_shape[0]
    w = inp_shape[1]
    max_h = int(h/4)
    max_w = int(w/4)

    # angle = random.random() * math.pi
    angle = random.randint(0, 360)
    scale = random.randint(80, 120) / 100
    #shift = random.random() * 0.8 * min(inp_img.shape[0], inp_img.shape[1])

    shift_y = random.randint(-max_h, max_h)
    shift_x = random.randint(-max_w, max_w)

    print(angle, shift_x, shift_y)

    center = (int(w/2), int(h/2))

    # поворот
    M = cv2.getRotationMatrix2D(center=center,
                                angle=angle,
                                scale=scale
                               )
    ret_img = cv2.warpAffine(inp_img, M, (inp_img.shape[1], inp_img.shape[0]),
                             #borderMode = cv2.BORDER_TRANSPARENT,
                             #borderValue = [255, 255, 255],
                            )

    # смещение
    M = np.float32([[1, 0, shift_y], [0, 1, shift_x]])
    ret_img = cv2.warpAffine(ret_img, M, (inp_img.shape[1], inp_img.shape[0]),
                             #borderMode = cv2.BORDER_TRANSPARENT,
                             #borderValue = [255, 255, 255],
                            )

    center_h = int(ret_img.shape[0] / 2)
    center_w = int(ret_img.shape[1] / 2)
    # вырезаю заданного размера
    if ret_img.shape[0] > h:
        ret_img = ret_img[center_h - int(h/2): center_h + int(h/2), :, :]

    if ret_img.shape[1] > w:
        ret_img = ret_img[:, center_w - int(w/2): center_w + int(w/2), :]

    return ret_img


In [49]:
def resize_to(inp_img: np.ndarray,
              inp_font: Optional[np.ndarray] = None,
             ) -> Tuple[np.ndarray, np.ndarray]:
    '''
    '''
    h, w = inp_img.shape[:2]

    if w >= h:
        devider = w / 640
    else:
        devider = h / 640

    new_w = int(w / devider)
    new_h = int(h / devider)

    aspect = w / h
    new_aspect = new_w / new_h
    print(aspect)
    print(new_aspect)

    ret_font = None
    ret_img = cv2.resize(inp_img, (new_w, new_h))
    
    if not isinstance(inp_font, type(None)):
        print('Change font too')
        ret_font = cv2.resize(inp_font, (new_w, new_h))

    return (ret_img, ret_font)


def apply_destruction(inp_img: np.ndarray, 
                      font_img: np.ndarray,
                      color: Optional[str] = 'white',
                     ) -> np.ndarray:
    '''
    '''
    if color == 'white':
        ret_img = cv2.bitwise_or(inp_img, font_img)
    else:
        ret_img = cv2.bitwise_not(inp_img)
        ret_img = cv2.bitwise_or(ret_img, font_img)
        ret_img = cv2.bitwise_not(ret_img)

    return ret_img


In [7]:
img2, _ = resize_to(img)
font2 = do_something(font, img2.shape)

1.5
1.5023474178403755
295 21 32


In [8]:
# tmp = apply_destruction(img2, font2)
tmp = apply_destruction(img2, font2, 'black')

In [9]:
cv2.imshow('current image', tmp)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [10]:
font_hsv = cv2.cvtColor(font2, cv2.COLOR_BGR2HSV)

In [11]:
cv2.imshow('current image', font2)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [63]:
iii = cv2.imread(os.path.join(DATA_PATH, '1.jpg'), cv2.IMREAD_UNCHANGED)
coffee = cv2.imread(os.path.join(FONT_PATH, 'cup_1.jpg'), cv2.IMREAD_UNCHANGED)

iii, coffee = resize_to(iii, coffee)

1.5005861664712778
1.5023474178403755
Change font too


In [64]:
hsv = cv2.cvtColor(coffee, cv2.COLOR_BGR2HSV)

lower_clr = np.array([0,0,130])
upper_clr = np.array([0,0,255])

mask = cv2.inRange(hsv, lower_clr, upper_clr)
mask = cv2.bitwise_not(mask)

In [32]:
mask.shape, mask.sum()

((336, 348), np.uint64(11203680))

In [21]:
hsv[0][0][2]

np.uint8(243)

In [68]:
tmp = cv2.bitwise_or(iii, coffee, mask=mask)
tmp_img = cv2.addWeighted(tmp, 0.9, iii, 0.1, 0.0)

In [70]:
cv2.imshow('current image', coffee)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
@app.route('/hello')
def hello():
    return 'Hello, World'

In [5]:
class PhotoRestoreaiton():

    def __init__(self, ):
        self._img_before_url = ''
        self._img_after_url = ''

        self._model = None

    
    def set_url_before(self, inp_url: str):
        self._img_before_url = inp_url

    def set_url_after(self, inp_url: str):
        self._img_after_url = inp_url
    
    def restore(self, ):
        pass


In [ ]:
    <p><input type="hidden" name="folder" size="200" /></p>

In [ ]:
def check_and_save_photo_before(inp_request) -> None:

    if 'file' not in inp_request.files:
        flash('No file part')
        return redirect(inp_request.url)

    file = inp_request.files['before_name']
    if file.filename == '':
        flash('No selected file')
        return redirect(inp_request.url)

    return 0